# Finish from version 4 — guided run only + test evaluation

Version 4 completed the **control** run and timed out on the **guided** run, then failed at
inference because the notebook passed `--device 0` instead of `--device cuda`. That script is
now fixed. This notebook:

1. restores `resume_state.zip` from the **version 4 output** (so the finished control run is kept),
2. resumes only the guided run to 150k,
3. evaluates both models once on the held-out LOL-v2-syn test set.

**Kaggle settings:** GPU T4 x2 (or any CUDA GPU), Internet ON.
**Input:** attach version 4's output as a dataset/input (`Add Data → Notebook Outputs → this notebook → version 4`).
It must contain `resume_state.zip`. Expected extra time: ~1–2 h.


In [ ]:
import os, sys, subprocess, glob, json, zipfile, shutil
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Crazycraftie/low-light-enhancement-btp.git', 'btp'], check=True)
os.chdir('/kaggle/working/btp')
print('Repo HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
if not Path('baselines/Retinexformer').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caiyuanhao1998/Retinexformer.git', 'baselines/Retinexformer'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'lmdb', 'yacs',
                'addict', 'future', 'natsort', 'yapf', 'thop', 'timm', 'lpips', 'scikit-image'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU (Accelerator must not be None).'
RUNS = Path('runs/post_midsem/final')
OPT = 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'
BEST_ALPHA = 4


## 1. Restore version 4 state (do not redo the completed control run)


In [ ]:
zips = glob.glob('/kaggle/input/**/resume_state.zip', recursive=True)
assert zips, ('Attach version 4 output as input; it must contain resume_state.zip. '
              'Use Add Data -> Notebook Outputs -> this notebook -> version 4.')
archive = max(zips, key=lambda p: os.path.getsize(p))
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/kaggle/working/btp')
print('Restored', archive)

for name in ('control', 'snr_attention_a4'):
    d = RUNS / name
    print(name, '| metrics.json:', (d / 'metrics.json').exists(),
          '| latest_state.pth:', (d / 'latest_state.pth').exists(),
          '| best.pth:', (d / 'best.pth').exists())
    if (d / 'run.log').exists():
        print('  last log line:', (d / 'run.log').read_text().strip().splitlines()[-1])
assert (RUNS / 'control' / 'metrics.json').exists(), 'Control run did not finish in version 4'
assert (RUNS / 'snr_attention_a4' / 'latest_state.pth').exists()


## 2. LOL-v2-syn data


In [ ]:
def gdl(fid, out, tries=4):
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown', '-q', fid, '-O', str(out)]).returncode == 0 and out.exists():
            return
    raise RuntimeError('Download failed: ' + str(out))

DATA = Path('data/LOLv2/Synthetic')
if not (DATA / 'Train/Low').is_dir():
    attached = glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low', recursive=True)
    if attached:
        source = Path(attached[0]).parents[2]
        Path('data').mkdir(exist_ok=True)
        if not Path('data/LOLv2').exists():
            Path('data/LOLv2').symlink_to(source, target_is_directory=True)
    else:
        gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U', '/kaggle/tmp/lolv2.zip')
        subprocess.run(['unzip', '-q', '-o', '/kaggle/tmp/lolv2.zip', '-d', 'data'], check=True)
for split, expected in (('Train/Low', 900), ('Test/Low', 100), ('Test/Normal', 100)):
    got = len(list((DATA / split).glob('*')))
    assert got == expected, f'{split}: {got} files, expected {expected}'
print('Data OK')


## 3. Resume only the guided run

Same arguments as the original notebook plus `--resume`. One GPU is enough here; timeout
allows 1.5 h, which is more than the remaining work.


In [ ]:
out = RUNS / 'snr_attention_a4'
if not (out / 'metrics.json').exists():
    common = ['--loss', 'l1', '--iters', '150000', '--lr', '2e-4', '--batch', '8',
              '--crop', '128', '--seed', '42', '--val-fraction', '0.1', '--val-seed', '2026',
              '--val-every', '5000', '--checkpoint-every', '5000', '--log-every', '500',
              '--data', str(DATA / 'Train'), '--opt', OPT, '--repo', 'baselines/Retinexformer',
              '--device', 'cuda', '--out', str(out), '--resume',
              '--snr-attention', '--alpha', str(BEST_ALPHA)]
    cmd = ['timeout', '5400', sys.executable, 'scripts/finetune.py'] + common
    print('Resuming guided run:', ' '.join(cmd[:6]), '...')
    with (out / 'run.log').open('a') as log_file:
        code = subprocess.run(cmd, stdout=log_file, stderr=subprocess.STDOUT).returncode
    print('exit code', code)
    if code not in (0, 124):
        raise RuntimeError((out / 'run.log').read_text()[-6000:])
    if code == 124:
        print('Timed out again: attach this version output and rerun.')
else:
    print('Guided run already complete')
with zipfile.ZipFile('/kaggle/working/resume_state.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUNS.rglob('*'):
        if p.is_file():
            zf.write(p, p.as_posix())
print('Saved /kaggle/working/resume_state.zip')


## 4. Test evaluation — both models, once

`--device cuda` (the old notebook used the invalid string `0`, now fixed in the script).


In [ ]:
def infer(weights, inp, output, method, alpha=None):
    cmd = [sys.executable, 'scripts/retinexformer_infer.py', '--repo', 'baselines/Retinexformer',
           '--opt', OPT, '--weights', str(weights), '--input', str(inp), '--output', str(output),
           '--method', method, '--device', 'cuda', '--no-speed-log']
    if alpha is not None:
        cmd += ['--snr-attention', '--alpha', str(alpha)]
    subprocess.run(cmd, check=True)

for name, alpha in (('control', None), ('snr_attention_a4', BEST_ALPHA)):
    run = RUNS / name
    if not (run / 'metrics.json').exists():
        print('Incomplete, skipping:', name)
        continue
    weights = run / 'best.pth'
    if not weights.exists():
        weights = run / 'final.pth'
    method = 'PostMidsem-control' if alpha is None else f'PostMidsem-SNR-a{alpha}'
    output = Path('results/post_midsem') / method / 'LOLv2-syn'
    infer(weights, DATA / 'Test/Low', output, method, alpha)
    for script, extra in (
        ('scripts/evaluate.py', ['--gt', str(DATA / 'Test/Normal')]),
        ('scripts/eval_dark_regions.py',
         ['--gt', str(DATA / 'Test/Normal'), '--low', str(DATA / 'Test/Low')]),
    ):
        subprocess.run([sys.executable, script, '--pred', str(output), '--method', method,
                        '--dataset', 'LOLv2-syn'] + extra, check=True)

with zipfile.ZipFile('/kaggle/working/final_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in (Path('results/results.csv'), Path('results/dark_region.csv')):
        if path.exists():
            zf.write(path, path.name)
    for root in (Path('results/per_image'), Path('results/per_image_dark')):
        for p in root.rglob('*'):
            if p.is_file() and 'PostMidsem' in str(p):
                zf.write(p, p.as_posix())
    for name in ('control', 'snr_attention_a4'):
        for filename in ('config.json', 'metrics.json', 'log.csv'):
            p = RUNS / name / filename
            if p.exists():
                zf.write(p, p.as_posix())
print('Saved /kaggle/working/final_results.zip')
print('\n'.join(l for l in Path('results/results.csv').read_text().splitlines()
                 if 'PostMidsem' in l))


## Done

Download `final_results.zip` from this version's Output and send it to me — I'll import the
CSVs and update the report, tables, and slides.
